In [1]:
import pandas as pd

# Đọc file CSV
true_df = pd.read_csv('News _dataset/True.csv')
fake_df = pd.read_csv('News _dataset/Fake.csv')

# Thêm nhãn
true_df['label'] = 1
fake_df['label'] = 0

# Gộp dữ liệu
data = pd.concat([true_df, fake_df], ignore_index=True)

# Xem thông tin
print(data.head())

                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  label  
0  December 31, 2017       1  
1  December 29, 2017       1  
2  December 31, 2017       1  
3  December 30, 2017       1  
4  December 29, 2017       1  


In [2]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # Xóa khoảng trắng thừa
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # Giữ lại chữ cái
    return text.strip().lower()

data['text'] = data['text'].apply(clean_text)

In [3]:
import pickle
with open('embeddings.pkl', 'rb') as f:
    data = pickle.load(f)

In [4]:
print(data.head(1))

                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   

                                                text       subject  \
0  washington  reuters    the head of a conservat...  politicsNews   

                 date  label  \
0  December 31, 2017       1   

                                           embedding  
0  [-0.28959557, -0.17193326, 0.22639382, 0.16168...  


In [5]:
# Lấy kích thước của một vector embedding
print(len(data['embedding'].iloc[0]))  # Kiểm tra số chiều của vector

768


In [6]:
# Hoặc kiểm tra tất cả các vector
all_lengths = data['embedding'].apply(len).unique()
print(f"Tất cả các vector đều có chiều: {all_lengths}")

Tất cả các vector đều có chiều: [768]


In [7]:
# Kiểm tra kiểu dữ liệu của một phần tử
print(type(data['embedding'].iloc[0]))  # Nên là list hoặc numpy.ndarray

<class 'numpy.ndarray'>


In [8]:
import numpy as np

# Chuyển cột embeddings thành mảng 2D NumPy
X = np.vstack(data['embedding'].values)

print(X.shape)  # Kích thước mảng: (số mẫu, 768)

(44898, 768)


In [9]:
from sklearn.decomposition import PCA

# Áp dụng PCA để giảm chiều
pca = PCA(n_components=100)  # Giảm xuống 100 chiều
X_reduced = pca.fit_transform(X)

print(X_reduced.shape)  # Kích thước mới: (số mẫu, 100)

(44898, 100)


In [10]:
import joblib
# Sau khi huấn luyện PCA
joblib.dump(pca, 'pca_model.pkl')

['pca_model.pkl']

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [12]:
# Giả sử cột 'label' chứa nhãn
y = data['label'].values

In [13]:
# Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

In [14]:
# Huấn luyện Gaussian Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

In [15]:
# Dự đoán và đánh giá
y_pred = model.predict(X_test)
print("Độ chính xác:", accuracy_score(y_test, y_pred))

Độ chính xác: 0.9024498886414254


In [16]:
from transformers import BertTokenizer, BertModel
import torch

# Load model và tokenizer một lần
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

c:\Users\Admin\OneDrive\Tài liệu\Fake_news_detection_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
def embed_text(text):
    # Tokenizer cho văn bản đơn lẻ
    tokens = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors='pt')  # Chạy trên CPU
    with torch.no_grad():
        outputs = bert_model(**tokens)
    # Trả về vector embedding (trung bình theo chiều thứ 1)
    return outputs.last_hidden_state.mean(dim=1).numpy()[0]  # [0] để lấy mảng 1D cho văn bản duy nhất

In [18]:
# Hàm dự đoán
def predict_fake_news(text):
    # Bước 1: Làm sạch văn bản
    cleaned_text = clean_text(text)
    
    # Bước 2: Chuyển đổi văn bản thành embedding
    embedding = embed_text(cleaned_text)  # Không dùng danh sách ở đây
    
    # Bước 3: Giảm chiều với PCA
    embedding_reduced = pca.transform([embedding])  # PCA yêu cầu đầu vào là mảng 2D
    
    # Bước 4: Dự đoán bằng model
    prediction = model.predict(embedding_reduced)
    
    # Bước 5: Trả kết quả
    return "True news" if prediction[0] == 1 else "Fake news"

In [19]:
# Ví dụ dự đoán
sample_text = "By now, the whole world knows that Alabama Senate candidate Roy Moore (R-Of Course) was banned from an Alabama mall and the YMCA for creeping on little girls. He has been accused of molesting young girls as young as age fourteen. Of course, when the allegations first came out and the uproar and backlash began, everyone, regardless of politics, reacted with outrage. However, the GOP s outrage was often much more muted. Further, it took awhile for the Republican National Committee to pull their support for Moore, despite the deeply disturbing allegations of his being a pedophile. Well, now that the rage has died down, the RNC is back with a new message regarding this Senate race: Better a pedophile than a Democrat.Under cover of night, the RNC reinstated their support for Roy Moore, and an RNC official confirmed to The Hill that, quote, We can confirm our involvement in the Alabama Senate race. So, there you have it, folks. They literally want a child molester in the United States Senate rather than a Democrat. There are only a few voices from the right who are being brave on this one   and none of them are seeking re-election. Former Republican presidential nominee Mitt Romney tweeted that the GOP must not tolerate Moore:Roy Moore in the US Senate would be a stain on the GOP and on the nation. Leigh Corfman and other victims are courageous heroes. No vote, no majority is worth losing our honor, our integrity.  Mitt Romney (@MittRomney) December 4, 2017Outgoing Arizona GOP Senator Jeff Flake has actually said that he would vote for the Democratic candidate, Doug Jones, if he lived in Alabama, and former Jeb Bush adviser Tim Miller has actually endorsed and donated to Jones. Outgoing Pennsylvania moderate Republican Charlie Dent said he never supported Moore in the first place. Here s the video of Dent s takedown of Moore:There is also outspoken Trump critic and longtime GOP strategist Steve Schmidt, who went on national television and called Moore a pedophile:Other than that, though, it has been radio silence. After all, they need that vote. Besides, let s face it   the GOP writ large showed America and the world what they stand for when they elected Donald Trump.Featured image via Scott Olson/Getty Images"
result = predict_fake_news(sample_text)
print("Dự đoán:", result)

Dự đoán: Fake news


In [20]:
import joblib
# Lưu mô hình Naive Bayes
joblib.dump(model, 'naive_bayes_model_using_BERT.pkl')

['naive_bayes_model_using_BERT.pkl']

In [21]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
# # Dự đoán nhãn của tập kiểm tra
# y_pred = model.predict(X_test)
# y_pred = (y_pred > 0.5).astype(int).flatten()  # Chuyển kết quả dự đoán thành nhãn 0 hoặc 1

# Tính toán các chỉ số đánh giá
print("Classification Report:")
print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      4650
           1       0.88      0.93      0.90      4330

    accuracy                           0.90      8980
   macro avg       0.90      0.90      0.90      8980
weighted avg       0.90      0.90      0.90      8980

Accuracy: 0.90
Precision: 0.88
Recall: 0.93
F1 Score: 0.90
